# Waveform shape vs 1/f normalisation: is the Iowa effect a relative-amplitude change or a non-sinusoidality effect?

**Why.** The harmonic check showed that the second harmonic of sharp-trough beta bursts falls in the 32–45 Hz flank
of the aperiodic (1/f) fit and inflates it. The primary burst amplitude is the envelope divided by that fit. In Iowa,
normalised burst amplitude is lower in PD (g ≈ −0.8) although the raw µV amplitude is not, and the aperiodic
slope is slightly flatter in PD in all three cohorts. Cortical beta in PD is known to be more non-sinusoidal
(Cole et al., 2017, J Neurosci 37:4830). So the effect could be (a) a genuine change in the
periodic-to-aperiodic balance or (b) sharper PD bursts whose harmonics raise their own normalisation floor.

**What this notebook does** (real data, all three cohorts, the pre-registered gate and component choice unchanged):

1. **Harmonic-safe normalisation.** Burst metrics are recomputed with the aperiodic fit over
   * `V0` 2–6.5 + 32–45 Hz (primary, as published);
   * `V1` 2–6.5 + **45–55 Hz** (above the second harmonic of beta ≤ 22 Hz, below the 60 Hz line);
   * `V2` 2–6.5 + 32–45 Hz **excluding ±3 Hz around twice the recording's beta peak**;
   * `V3` specparam/FOOOF aperiodic fit over 2–45 Hz with peaks modelled (only if `fooof` or `specparam` is installed);

   each with its own common threshold (75th percentile of the pooled normalised envelopes). Also: the gate re-run with
   the `V1` fit (`V1_regate`), to see whether the harmonic changes who is included.
2. **Waveform shape** of the chosen GED component inside the detected bursts (Cole et al., 2017): extrema sharpness
   asymmetry |log₁₀(peak/trough sharpness)| and rise–decay asymmetry |log₁₀(rise/decay time)|, both
   polarity-invariant (the sign of a GED component is arbitrary); the **harmonic excess** (dB above the harmonic-safe
   1/f fit at 2 × beta peak ± 2 Hz); and the **floor inflation** 10·log₁₀(aperiodic beta power V0 / V1), i.e. how
   much the 32–45 Hz flank raises the normalisation floor.
3. **Tests:** PD vs HC per cohort and random-effects meta-analysis for every variant and waveform measure; OFF vs ON for
   the waveform measures; Iowa regression of normalised amplitude on group with and without the waveform measures as
   covariates; correlations of the floor inflation with harmonic excess and sharpness.

**How to read the result**
* Iowa g stays ≈ −0.8 under V1/V2/V3, and adding the waveform covariates leaves the PD coefficient intact →
  the relative-amplitude effect stands; the harmonic confound is ruled out.
* g shrinks under V1/V2/V3, floor inflation and harmonic excess are higher in PD, and the covariates absorb the
  PD coefficient → the effect is a **waveform-shape** effect expressed through the normalisation.
* The flatter PD slope under V0 vanishes under V1 → the "aperiodic" difference was harmonic energy, not 1/f.

Put this notebook next to `revision_analyses.ipynb`. Runtime about 15–25 min (EEG loaded once per cohort).
Outputs: `pd_beta_three/revision/W1_*.csv`.

In [1]:
"""Settings."""
W_DATASETS = None                 # None = every cohort of the main notebook; or e.g. ["iowa"]
HARMONIC_FLANK = (45.0, 55.0)     # V1 upper flank (Hz); data were notch-filtered at 60 Hz and low-passed at 100 Hz
HARMONIC_EXCLUDE_HZ = 3.0         # V2: half-width of the band excluded around 2 × beta peak
SHARP_MS = 5.0                    # sharpness lag (Cole et al. 2017 use ≈ 5 ms)
WAVE_BAND = (2.0, 45.0)           # broadband signal on which extrema and sharpness are measured
REVISION_NOTEBOOK = "revision_analyses.ipynb"

In [2]:
"""Load the code of the revision notebook (settings, loader, R4 functions; nothing heavy runs)."""
import json
from pathlib import Path

_nb = json.loads(Path(REVISION_NOTEBOOK).read_text(encoding="utf-8"))
_want = ('"""Settings."""', '"""Load every definition', "def candidates(")
for _cell in _nb["cells"]:
    _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]
    if _cell["cell_type"] == "code" and _src.lstrip().startswith(_want):
        exec(compile(_src, REVISION_NOTEBOOK, "exec"), globals())
_SPECTRAL_V0 = spectral
W_DATASETS = [d for d in (W_DATASETS or list(ANAT)) if d in ANAT]
try:
    from fooof import FOOOF as _SpecModel
    SPEC_LIB = "fooof"
except ImportError:
    try:
        from specparam import SpectralModel as _SpecModel
        SPEC_LIB = "specparam"
    except ImportError:
        _SpecModel, SPEC_LIB = None, None
print(f"Cohorts: {[LABEL[d] for d in W_DATASETS]}; V3 (spectral parameterisation): {SPEC_LIB or 'not installed, skipped'}")

Loaded pd_beta_three_datasets.ipynb: datasets ['sandiego', 'newmexico', 'iowa'], head model fsaverage, outputs → pd_beta_three\revision
  San Diego: 32 channels, simulated duration 183 s
  New Mexico: 60 channels, simulated duration 58 s
  Iowa: 58 channels, simulated duration 132 s
Cohorts: ['San Diego', 'New Mexico', 'Iowa']; V3 (spectral parameterisation): fooof


C:\Users\RTX2080Ti\AppData\Local\Temp\ipykernel_3532\324823077.py:14: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  from fooof import FOOOF as _SpecModel


In [3]:
TRAP = getattr(np, "trapezoid", None) or np.trapz


def ap_fit(sig, valid, fs, hi=(32.0, 45.0), fmax=45.0, exclude=None):
    """Same model as spectral() (log-log line through 2–6.5 Hz and an upper flank); optional excluded band."""
    f, p = signal.welch(sig[valid], fs, nperseg=int(2 * fs))
    sel = (f >= 2) & (f <= fmax)
    f, p = f[sel], p[sel]
    flank = ((f >= 2) & (f <= 6.5)) | ((f >= hi[0]) & (f <= hi[1]))
    if exclude is not None:
        flank &= ~((f >= exclude[0]) & (f <= exclude[1]))
    coef = np.polyfit(np.log10(f[flank]), np.log10(p[flank]), 1)
    fit = np.polyval(coef, np.log10(f))
    resid = 10 * (np.log10(p) - fit)
    b = (f >= 13) & (f <= 30)
    j = np.argmax(resid[b])
    return {"beta_db": float(resid[b][j]), "peak_hz": float(f[b][j]), "slope": float(coef[0]),
            "ap_var": float(TRAP(10 ** fit[b], f[b])), "curve": (f, 10 * np.log10(p), 10 * fit)}


def ap_specparam(sig, valid, fs):
    if _SpecModel is None:
        return None
    f, p = signal.welch(sig[valid], fs, nperseg=int(2 * fs))
    try:
        fm = _SpecModel(peak_width_limits=[1.0, 8.0], max_n_peaks=6, aperiodic_mode="fixed", verbose=False)
        fm.fit(f, p, [2.0, 45.0])
        off, expo = (getattr(fm, "aperiodic_params_", None) if hasattr(fm, "aperiodic_params_")
                     else fm.get_params("aperiodic"))[:2]
    except Exception:
        return None
    b = (f >= 13) & (f <= 30)
    return {"slope": -float(expo), "ap_var": float(TRAP(10 ** (off - expo * np.log10(f[b])), f[b]))}


def variant_fits(x, valid, fs):
    v0 = _SPECTRAL_V0(x, valid, fs)
    out = {"V0": v0, "V1": ap_fit(x, valid, fs, HARMONIC_FLANK, HARMONIC_FLANK[1])}
    h = 2 * v0["peak_hz"]
    out["V2"] = ap_fit(x, valid, fs, exclude=(h - HARMONIC_EXCLUDE_HZ, h + HARMONIC_EXCLUDE_HZ))
    v3 = ap_specparam(x, valid, fs)
    if v3 is not None:
        out["V3"] = v3
    return out


def waveform(x, valid, events, fs):
    """Polarity-invariant shape measures of the component inside bursts (extrema from beta zero crossings)."""
    xb = sim.bandpass(x, WAVE_BAND, fs)
    xn = sim.bandpass(x, CFG["beta_band"], fs)
    inb = np.zeros(len(x), bool)
    for e in events:
        inb[e["onset"]:e["offset"]] = True
    inb &= valid
    k = max(1, int(round(SHARP_MS / 1000 * fs)))
    s = np.sign(xn)
    rise = np.where((s[:-1] < 0) & (s[1:] >= 0))[0] + 1
    fall = np.where((s[:-1] > 0) & (s[1:] <= 0))[0] + 1
    zc = sorted([(i, "r") for i in rise] + [(i, "f") for i in fall])
    ext = []                                                   # (time, +1 peak / −1 trough)
    for (a, ta), (b, tb) in zip(zc[:-1], zc[1:]):
        if b - a < 2 or not inb[a:b].all() or a < k or b > len(x) - k:
            continue
        if ta == "r" and tb == "f":
            ext.append((a + int(np.argmax(xb[a:b])), 1))
        elif ta == "f" and tb == "r":
            ext.append((a + int(np.argmin(xb[a:b])), -1))
    if len(ext) < 20:
        return {"n_extrema": len(ext)}
    t = np.array([e[0] for e in ext]); typ = np.array([e[1] for e in ext])
    sharp = (np.abs(xb[t] - xb[t - k]) + np.abs(xb[t] - xb[t + k])) / 2
    ps, ts = sharp[typ == 1].mean(), sharp[typ == -1].mean()
    rises, decays = [], []
    for (t0, y0), (t1, y1) in zip(ext[:-1], ext[1:]):
        if y0 == y1 or not inb[t0:t1].all():
            continue
        (rises if y0 == -1 else decays).append(t1 - t0)
    out = {"n_extrema": len(ext), "sharp_asym": abs(np.log10(ps / ts)),
           "rd_asym": abs(np.log10(np.mean(rises) / np.mean(decays))) if rises and decays else np.nan}
    return out


def harmonic_excess(x, valid, fs, f0):
    """Mean power (dB) above the harmonic-safe 1/f fit within 2·f0 ± 2 Hz (NaN if 2·f0 is outside 32–53 Hz)."""
    h = 2 * f0
    if h - 2 < 32 or h + 2 > HARMONIC_FLANK[1] - 2:
        return np.nan
    f, p_db, fit_db = ap_fit(x, valid, fs, HARMONIC_FLANK, HARMONIC_FLANK[1], exclude=(h - 2, h + 2))["curve"]
    sel = (f >= h - 2) & (f <= h + 2)
    return float(np.mean(p_db[sel] - fit_db[sel]))


def analyse_cohort(ds):
    f = REV / f"W1_per_recording_{ds}.csv"
    if f.exists():
        return pd.read_csv(f), json.loads((REV / f"W1_regate_{ds}.json").read_text())
    t0 = time.time()
    D = load_real(ds)
    chosen = {s: select(c, GATE) for s, c in D["cands"].items()}
    inc = [s for s, (c, ok) in chosen.items() if ok]
    keys = [(s, c) for s in inc for c in D["conds"][s]]
    base, sbs = {}, {}
    for key in keys:
        d = D["recs"][key]
        x = chosen[key[0]][0]["w"] @ d["X"]
        fits = variant_fits(x, d["valid"], SFREQ)
        sbeta = sim.bandpass(x, CFG["beta_band"], SFREQ)
        sbs[key] = {v: sbeta / np.sqrt(fits[v]["ap_var"]) for v in fits}
        base[key] = {"x": x, "fits": fits}
    variants = sorted({v for k in keys for v in sbs[k]})
    rows = {k: {"dataset": ds, "subject": k[0], "condition": k[1], "stem": D["recs"][k]["stem"],
                "group": D["recs"][k]["group"]} for k in keys}
    for vname in variants:
        ks = [k for k in keys if vname in sbs[k]]
        thr = float(np.percentile(np.concatenate([np.abs(signal.hilbert(sbs[k][vname]))[D["recs"][k]["valid"]] for k in ks]),
                                  THRESH_PERCENTILE))
        for k in ks:
            v = D["recs"][k]["valid"]
            ev, env = detect_real(sbs[k][vname], v, SFREQ, thr)
            st = burst_stats(ev, env, v, SFREQ)
            fit = base[k]["fits"][vname]
            rows[k].update({f"burst_amp_{vname}": st["burst_amp"], f"rate_hz_{vname}": st["rate_hz"],
                            f"slope_{vname}": fit["slope"], f"ap_db_{vname}": 10 * np.log10(fit["ap_var"])})
            if vname == "V0":
                rows[k].update(waveform(base[k]["x"], v, ev, SFREQ))
                rows[k]["harmonic_db"] = harmonic_excess(base[k]["x"], v, SFREQ, fit["peak_hz"])
                rows[k]["beta_peak_hz"] = fit["peak_hz"]
                rows[k]["beta_db_V0"] = fit["beta_db"]
        for k in ks:
            rows[k][f"thr_{vname}"] = thr
    for k in keys:
        rows[k]["floor_inflation_db"] = rows[k]["ap_db_V0"] - rows[k]["ap_db_V1"]
        rows[k]["beta_db_V1"] = base[k]["fits"]["V1"]["beta_db"]
    res = pd.DataFrame(rows.values())
    # gate re-run with the harmonic-safe fit: who would be included?
    global spectral
    spectral = lambda sig, valid, fs: ap_fit(sig, valid, fs, HARMONIC_FLANK, HARMONIC_FLANK[1])
    try:
        rg, sb1 = {"V0": {}, "V1": {}}, {}
        for s, cs in D["conds"].items():
            Xs, vs = [D["recs"][(s, c)]["X"] for c in cs], [D["recs"][(s, c)]["valid"] for c in cs]
            c1, ok1 = select(candidates(Xs, vs, D["chans"]), GATE)
            grp = D["recs"][(s, cs[0])]["group"]
            rg["V0"].setdefault(grp, [0, 0]); rg["V1"].setdefault(grp, [0, 0])
            rg["V0"][grp][0] += int(chosen[s][1]); rg["V0"][grp][1] += 1
            rg["V1"][grp][0] += int(ok1); rg["V1"][grp][1] += 1
            if ok1:
                for c in cs:
                    d = D["recs"][(s, c)]
                    sb, sp = normalised_beta(d["X"], d["valid"], c1["w"], SFREQ)
                    res.loc[(res.subject == s) & (res.condition == c), "regate_component_same"] = \
                        bool(chosen[s][1] and c1["label"] == chosen[s][0]["label"])
                    sb1[(s, c)] = sb
        extra = []
        if sb1:
            thr1 = float(np.percentile(np.concatenate([np.abs(signal.hilbert(sb))[D["recs"][k]["valid"]]
                                                       for k, sb in sb1.items()]), THRESH_PERCENTILE))
        for (s, c), sb in sb1.items():
            d = D["recs"][(s, c)]
            ev, env = detect_real(sb, d["valid"], SFREQ, thr1)
            extra.append({"dataset": ds, "subject": s, "condition": c, "stem": d["stem"], "group": d["group"],
                          "burst_amp_V1_regate": burst_stats(ev, env, d["valid"], SFREQ)["burst_amp"]})
        pd.DataFrame(extra).to_csv(REV / f"W1_regate_recordings_{ds}.csv", index=False)
    finally:
        spectral = _SPECTRAL_V0
    res.to_csv(f, index=False)
    (REV / f"W1_regate_{ds}.json").write_text(json.dumps(rg))
    print(f"  {LABEL[ds]}: {len(res)} recordings, {time.time() - t0:.0f} s")
    del D
    gc.collect()
    return res, rg

In [4]:
W, REG = {}, {}
for ds in W_DATASETS:
    print(f"{LABEL[ds]} …")
    W[ds], REG[ds] = analyse_cohort(ds)
WA = pd.concat(W.values(), ignore_index=True)
ref = pd.concat([per_rec(ds)[["stem", "burst_amp"]].assign(dataset=ds) for ds in W_DATASETS if per_rec(ds) is not None])
chk = WA.merge(ref, on=["dataset", "stem"])
print(f"Check: V0 reproduces the published burst amplitude, r = {np.corrcoef(chk.burst_amp_V0, chk.burst_amp)[0, 1]:.4f} "
      f"({len(chk)} recordings)")

San Diego …
  San Diego: 37 recordings, 39 s
New Mexico …
  New Mexico: 58 recordings, 53 s
Iowa …
  Iowa: 80 recordings, 314 s
Check: V0 reproduces the published burst amplitude, r = 1.0000 (175 recordings)


In [5]:
"""1. PD vs HC for every normalisation variant and waveform measure, and the meta-analysis."""
VARS = [c for c in ("V0", "V1", "V2", "V3") if f"burst_amp_{c}" in WA]
AMPS = [f"burst_amp_{v}" for v in VARS] + [f"rate_hz_{v}" for v in VARS]
SPEC = [f"slope_{v}" for v in VARS] + [f"ap_db_{v}" for v in VARS] + ["floor_inflation_db", "beta_db_V0", "beta_db_V1"]
SHAPE = ["sharp_asym", "rd_asym", "harmonic_db", "beta_peak_hz"]
fam = {"amplitude": AMPS, "spectrum": SPEC, "shape": SHAPE}
tabs = []
for ds in W_DATASETS:
    for name, ms in fam.items():
        t = groups(W[ds], ms, ds)
        if len(t):
            tabs.append(t.assign(family=name))
G = save(pd.concat(tabs, ignore_index=True), "W1_pd_vs_hc")
print("PD vs HC (Hedges g; Holm within cohort × contrast × family):")
show(G, ["dataset", "contrast", "family", "metric", "n_PD", "n_HC", "PD", "HC", "g", "p", "p_holm"], nd=3)

M = []
for name, ms in fam.items():
    g_ = G[(G.contrast == "PD-ON vs HC") & G.metric.isin(ms)]
    if g_.dataset.nunique() > 1:
        M.append(meta(g_, "g").assign(family=name))
if M:
    M = save(pd.concat(M, ignore_index=True), "W1_meta")
    print("\nMeta-analysis of PD-ON vs HC across cohorts (fixed effect and random effects, HKSJ CI):")
    show(M, ["family", "metric", "k", "pooled", "ci_low", "ci_high", "p", "I2", "tau2", "RE_pooled", "RE_HKSJ_low",
             "RE_HKSJ_high", "RE_HKSJ_p"], nd=3)

PD vs HC (Hedges g; Holm within cohort × contrast × family):
  dataset     contrast    family             metric  n_PD  n_HC     PD     HC      g     p  p_holm
 sandiego  PD-ON vs HC amplitude       burst_amp_V0    14     9  3.670  3.738 -0.211 0.925   1.000
 sandiego  PD-ON vs HC amplitude       burst_amp_V1    14     9  3.780  3.901 -0.308 0.729   1.000
 sandiego  PD-ON vs HC amplitude       burst_amp_V2    14     9  3.652  3.720 -0.219 0.777   1.000
 sandiego  PD-ON vs HC amplitude       burst_amp_V3    14     9  3.629  3.698 -0.222 0.925   1.000
 sandiego  PD-ON vs HC amplitude         rate_hz_V0    14     9  0.887  0.862  0.055 0.777   1.000
 sandiego  PD-ON vs HC amplitude         rate_hz_V1    14     9  0.863  0.833  0.063 0.975   1.000
 sandiego  PD-ON vs HC amplitude         rate_hz_V2    14     9  0.877  0.868  0.020 0.925   1.000
 sandiego  PD-ON vs HC amplitude         rate_hz_V3    14     9  0.890  0.857  0.071 0.729   1.000
 sandiego PD-OFF vs HC amplitude       burst_amp

In [6]:
"""2. OFF vs ON for the waveform measures (does dopamine change the shape?)."""
P = [paired(W[ds], SHAPE + ["floor_inflation_db", "burst_amp_V1"], ds) for ds in W_DATASETS
     if (W[ds].condition == "off").any()]
P = [p for p in P if len(p)]
if P:
    P = save(pd.concat(P, ignore_index=True), "W1_off_on")
    show(P, ["dataset", "metric", "n_pairs", "OFF", "ON", "d_z", "p", "p_holm"], nd=3)

  dataset             metric  n_pairs    OFF     ON    d_z     p  p_holm
 sandiego         sharp_asym       14  0.034  0.021  0.499 0.119   0.713
 sandiego            rd_asym       14  0.015  0.010  0.356 0.119   0.713
 sandiego        harmonic_db        9  0.315  0.090  0.394 0.359   1.000
 sandiego       beta_peak_hz       14 21.000 21.179 -0.045 0.648   1.000
 sandiego floor_inflation_db       14  0.212  0.203  0.026 0.903   1.000
 sandiego       burst_amp_V1       14  3.725  3.780 -0.206 0.502   1.000
newmexico         sharp_asym       19  0.034  0.039 -0.178 0.241   0.965
newmexico            rd_asym       19  0.020  0.024 -0.199 0.768   1.000
newmexico        harmonic_db       13 -0.012  0.332 -0.311 0.414   1.000
newmexico       beta_peak_hz       20 19.525 19.525  0.000 0.507   1.000
newmexico floor_inflation_db       20 -0.070  0.143 -0.436 0.105   0.527
newmexico       burst_amp_V1       20  4.058  4.212 -0.458 0.076   0.455


In [7]:
"""3. Iowa: does waveform shape explain the group effect? (OLS, standardised PD coefficient)"""
if "iowa" in W:
    io = with_covariates(W["iowa"], "iowa")
    base_cov = [c for c in ("age", "male", "moca") if c in io and io[c].notna().any()]
    models = [("V0, demographics", "burst_amp_V0", base_cov),
              ("V0, + sharpness + rise/decay", "burst_amp_V0", base_cov + ["sharp_asym", "rd_asym"]),
              ("V0, + harmonic excess", "burst_amp_V0", base_cov + ["harmonic_db"]),
              ("V0, + floor inflation", "burst_amp_V0", base_cov + ["floor_inflation_db"]),
              ("V0, + all shape measures", "burst_amp_V0", base_cov + ["sharp_asym", "rd_asym", "harmonic_db", "floor_inflation_db"]),
              ("V1, demographics", "burst_amp_V1", base_cov),
              ("V2, demographics", "burst_amp_V2", base_cov)]
    if "burst_amp_V3" in io:
        models.append(("V3, demographics", "burst_amp_V3", base_cov))
    out = []
    for lab, y, cov in models:
        b, se, p, n = ols_group(io, y, cov)
        sd = io[y].std()
        out.append({"model": lab, "outcome": y, "n": n, "beta_std_PD": b / sd, "ci_low": (b - 1.96 * se) / sd,
                    "ci_high": (b + 1.96 * se) / sd, "p": p})
    O = save(pd.DataFrame(out), "W1_iowa_models")
    print(f"Iowa PD − HC (β / SD of outcome), covariates {', '.join(base_cov)} plus those named:")
    show(O, nd=3)
    print("\nIowa, across recordings (Spearman ρ):")
    pairs = [("harmonic_db", "floor_inflation_db"), ("sharp_asym", "floor_inflation_db"), ("rd_asym", "floor_inflation_db"),
             ("floor_inflation_db", "burst_amp_V0"), ("harmonic_db", "burst_amp_V0"), ("sharp_asym", "burst_amp_V0"),
             ("burst_amp_V0", "burst_amp_V1")]
    cr = []
    for a_, b_ in pairs:
        dd = io[[a_, b_]].dropna()
        r, p = stats.spearmanr(dd[a_], dd[b_])
        cr.append({"x": a_, "y": b_, "n": len(dd), "rho": r, "p": p})
    show(save(pd.DataFrame(cr), "W1_iowa_correlations"), nd=3)

Iowa PD − HC (β / SD of outcome), covariates age, male, moca plus those named:
                       model      outcome  n  beta_std_PD  ci_low  ci_high     p
            V0, demographics burst_amp_V0 80       -0.836  -1.290   -0.383 0.001
V0, + sharpness + rise/decay burst_amp_V0 75       -0.678  -1.091   -0.265 0.002
       V0, + harmonic excess burst_amp_V0 63       -0.805  -1.208   -0.402 0.000
       V0, + floor inflation burst_amp_V0 80       -0.729  -1.071   -0.386 0.000
    V0, + all shape measures burst_amp_V0 60       -0.546  -0.869   -0.222 0.002
            V1, demographics burst_amp_V1 79       -0.764  -1.226   -0.301 0.002
            V2, demographics burst_amp_V2 80       -0.851  -1.304   -0.398 0.000
            V3, demographics burst_amp_V3 80       -0.839  -1.294   -0.385 0.001

Iowa, across recordings (Spearman ρ):
                 x                  y  n    rho     p
       harmonic_db floor_inflation_db 63  0.726 0.000
        sharp_asym floor_inflation_db 75  0.3

In [8]:
"""4. The gate re-run with the harmonic-safe fit."""
rows = []
for ds, rg in REG.items():
    for grp in ("PD", "HC"):
        if grp in rg.get("V0", {}):
            rows.append({"dataset": LABEL[ds], "group": grp, "included_V0": f"{rg['V0'][grp][0]}/{rg['V0'][grp][1]}",
                         "included_V1_gate": f"{rg['V1'][grp][0]}/{rg['V1'][grp][1]}"})
show(save(pd.DataFrame(rows), "W1_regate_counts"))
rr = [pd.read_csv(f) for ds in W_DATASETS for f in [REV / f"W1_regate_recordings_{ds}.csv"] if f.exists()]
if rr:
    RR = pd.concat(rr, ignore_index=True)
    t = pd.concat([groups(RR[RR.dataset == ds], ["burst_amp_V1_regate"], ds) for ds in RR.dataset.unique()])
    print("\nPD vs HC, burst amplitude with the harmonic-safe fit used for the gate AND the normalisation:")
    show(save(t, "W1_regate_pd_vs_hc"), ["dataset", "contrast", "n_PD", "n_HC", "PD", "HC", "g", "p"], nd=3)
    if t.dataset.nunique() > 1:
        show(meta(t[t.contrast == "PD-ON vs HC"], "g"), ["metric", "k", "pooled", "ci_low", "ci_high", "p", "I2", "tau2",
                                                          "RE_pooled", "RE_HKSJ_low", "RE_HKSJ_high", "RE_HKSJ_p"], nd=3)

   dataset group included_V0 included_V1_gate
 San Diego    PD       14/14            14/14
 San Diego    HC        9/11             9/11
New Mexico    PD       22/24            22/24
New Mexico    HC       16/18            16/18
      Iowa    PD       48/87            45/87
      Iowa    HC       32/44            31/44

PD vs HC, burst amplitude with the harmonic-safe fit used for the gate AND the normalisation:
  dataset     contrast  n_PD  n_HC    PD    HC      g     p
 sandiego  PD-ON vs HC    14     9 3.862 3.948 -0.206 0.875
 sandiego PD-OFF vs HC    14     9 3.800 3.948 -0.342 0.329
newmexico  PD-ON vs HC    22    16 4.207 4.190  0.038 0.871
newmexico PD-OFF vs HC    20    16 4.062 4.190 -0.277 0.258
     iowa  PD-ON vs HC    45    31 3.867 4.177 -0.790 0.002
             metric  k  pooled  ci_low  ci_high     p    I2  tau2  RE_pooled  RE_HKSJ_low  RE_HKSJ_high  RE_HKSJ_p
burst_amp_V1_regate  3  -0.448  -0.796   -0.101 0.011 0.556 0.132     -0.369       -1.488         0.751     

In [9]:
io = with_covariates(W["iowa"], "iowa")
shape = ["sharp_asym", "rd_asym", "harmonic_db", "floor_inflation_db"]
cc = io.dropna(subset=shape)
for lab, cov in [("demographics only", base_cov), ("+ all shape", base_cov + shape)]:
    b, se, p, n = ols_group(cc, "burst_amp_V0", cov)
    sd = cc["burst_amp_V0"].std()
    print(f"{lab:18s} n={n}  beta_std={b/sd:.3f} [{(b-1.96*se)/sd:.3f}, {(b+1.96*se)/sd:.3f}]  p={p:.4f}")

demographics only  n=60  beta_std=-0.875 [-1.373, -0.376]  p=0.0011
+ all shape        n=60  beta_std=-0.576 [-0.917, -0.234]  p=0.0017
